# Attribution Patching Recipe: the IOI circuit of GPT-2 small from one gradient pass

Activation patching asks, for every component, what happens to a metric when that component's activation is swapped in from a corrupted run. Done literally, that is one forward pass per edge of the model. Attribution patching replaces the sweep with a first-order estimate from one clean forward, one corrupted forward and one backward pass (Syed, Rager and Conmy 2023, "Attribution Patching Outperforms Automated Circuit Discovery", arXiv:2310.10348). EAP-IG integrates the gradient along the path from the corrupted to the clean input, which repairs the estimate where the gradient saturates (Hanna, Pezzelle and Belinkov 2024, "Have Faith in Faithfulness: Going Beyond Circuit Overlap When Finding Model Mechanisms", arXiv:2403.17806).

This notebook scores every edge of GPT-2 small's component graph on the indirect-object identification task (Wang et al. 2022, "Interpretability in the Wild", arXiv:2211.00593), reads the circuit off the ranking, and checks the circuit's faithfulness by patching every other edge in one call. It downloads GPT-2 small (about 550 MB) at a pinned revision and runs on a laptop CPU in about ten minutes, most of it in the faithfulness replays.

In [ ]:
from __future__ import annotations

import time

import torch
from transformers import AutoTokenizer, GPT2LMHeadModel

import torchlens as tl
import torchlens.mechinterp

MODEL_ID = "openai-community/gpt2"
REVISION = "607a30d783dfa663caf39e06633721c8d4cfcd7e"  # pragma: allowlist secret
torch.manual_seed(0)
model = GPT2LMHeadModel.from_pretrained(
    MODEL_ID, revision=REVISION, attn_implementation="eager", use_cache=False
).eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)

## The task: clean IOI prompts and their ABC corruptions

Each clean prompt names two people, repeats one of them (the subject, S) and stops where the model should name the other (the indirect object, IO). The corrupted prompt replaces the repeated name with a third one, so the clean answer is no longer determined. Clean and corrupted prompts match token for token; the metric is the IO-minus-S logit difference at the last position.

In [ ]:
TEMPLATE = "When {A} and {B} went to the store, {S} gave a drink to"
TRIPLES = [
    ("Mary", "John", "Tom"),
    ("Alice", "Bob", "Carl"),
    ("Sarah", "Mike", "Paul"),
    ("Anna", "Peter", "Jack"),
]


def single_token(name: str) -> int:
    """Return the one GPT-2 token of a leading-space name."""
    ids = tokenizer.encode(" " + name)
    assert len(ids) == 1, name
    return ids[0]


clean_texts, corrupt_texts, io_tokens, s_tokens = [], [], [], []
for a, b, c in TRIPLES:
    for first, second in ((a, b), (b, a)):  # both name orders
        clean_texts.append(TEMPLATE.format(A=first, B=second, S=second))
        corrupt_texts.append(TEMPLATE.format(A=first, B=second, S=c))
        io_tokens.append(single_token(first))
        s_tokens.append(single_token(second))
clean_ids = torch.tensor([tokenizer.encode(text) for text in clean_texts])
corrupt_ids = torch.tensor([tokenizer.encode(text) for text in corrupt_texts])
assert clean_ids.shape == corrupt_ids.shape
io_ids, s_ids = torch.tensor(io_tokens), torch.tensor(s_tokens)
rows = torch.arange(len(clean_texts))


def logit_diff(logits: torch.Tensor) -> torch.Tensor:
    """IO-minus-S logit at the last position, one value per prompt."""
    last = logits[:, -1]
    return last[rows, io_ids] - last[rows, s_ids]


with torch.no_grad():
    clean_metric = logit_diff(model(clean_ids).logits)
    corrupt_metric = logit_diff(model(corrupt_ids).logits)
print(f"logit difference: clean {clean_metric.mean():+.2f}, corrupted {corrupt_metric.mean():+.2f}")
assert clean_metric.mean() > corrupt_metric.mean()

## Score every edge with EAP-IG

`edge_scores` builds the component graph (the embedding, every attention head, every head bias, every MLP and the logits, with each head's q, k and v inputs as separate receivers) and returns one score per edge from one backward pass per integration step. A score estimates how much the metric moves when that edge alone is patched to its corrupted value, so the most negative scores are the edges the task depends on.

In [ ]:
started = time.perf_counter()
table = torchlens.mechinterp.edge_scores(
    model, clean_ids, corrupted_input=corrupt_ids, metric=logit_diff, method="eap_ig", steps=5
)
scores = {
    address[0]: row.score for (_target, kind, address), row in table.items() if kind == "EDGE"
}
ranked = sorted(scores, key=lambda edge: -abs(scores[edge]))
print(f"{len(scores)} edges scored in {time.perf_counter() - started:.0f} s; the 15 largest:")
for edge in ranked[:15]:
    print(f"  {edge:>22}  {scores[edge]:+.3f}")

The paper's circuit is readable from the top of the list: the name-mover heads 9.9, 9.6 and 10.0 writing into the logits, and the S-inhibition heads of layers 7 and 8 feeding the name movers' queries.

In [ ]:
NAME_MOVER_EDGES = {"a9.h9->logits", "a9.h6->logits", "a10.h0->logits"}
found = sorted(NAME_MOVER_EDGES & set(ranked[:50]))
print("name-mover edges in the top 50:", found)
assert found, "the IOI name movers should score among the top edges"

## Check the circuit: keep the top edges, patch everything else in one call

A circuit is faithful when the model with every edge outside it patched to its corrupted value still recovers the clean metric. `patch_edges` applies all those patches in one replay of a replayable clean trace. Faithfulness is the recovered fraction of the clean-minus-corrupted metric (the MIB convention), and `circuit_performance_ratio` is the area under that curve over the kept fractions.

In [ ]:
ready = tl.options.CaptureOptions(replayable=True)
clean_trace = tl.trace(model, clean_ids, capture=ready, save="all")
corrupt_trace = tl.trace(model, corrupt_ids, capture=ready, save="all")
graph = torchlens.mechinterp.component_graph(clean_trace)
corrupt_graph = torchlens.mechinterp.component_graph(corrupt_trace)
full, baseline = clean_metric.mean().item(), corrupt_metric.mean().item()


def faithfulness(kept: set[str]) -> float:
    """Patch every edge outside ``kept`` to its corrupted value; return the recovered fraction."""
    pruned = [edge for name, edge in graph.edges.items() if name not in kept]
    patched = torchlens.mechinterp.patch_edges(clean_trace, pruned, corrupted_graph=corrupt_graph)
    value = logit_diff(patched.modules["lm_head"].out).mean().item()
    return (value - baseline) / (full - baseline)


fractions = [0.002, 0.01, 0.05, 0.1]
curve = {}
started = time.perf_counter()
for fraction in fractions:
    kept = set(ranked[: max(1, round(fraction * len(ranked)))])
    curve[fraction] = faithfulness(kept)
    print(f"keep {fraction:>6.1%} ({len(kept):>5} edges): faithfulness {curve[fraction]:+.2f}")
print(f"{len(fractions)} patched replays in {time.perf_counter() - started:.0f} s")
cpr = torchlens.mechinterp.circuit_performance_ratio(curve)
print("circuit performance ratio (area under the curve):", round(cpr, 4))
assert curve[0.1] > 0.5, "ten percent of the edges should recover most of the clean behaviour"